# MCQ Verification — Datathon 2026 Round 1
**10 câu trắc nghiệm — Verify từng câu trực tiếp từ data, không đoán mò.**

| Câu | Nội dung | File chính |
|-----|---------|-----------|
| Q1 | Median inter-order gap | orders.csv |
| Q2 | Segment có gross margin cao nhất | products.csv |
| Q3 | Lý do trả hàng phổ biến nhất cho Streetwear | returns + products |
| Q4 | Traffic source có bounce_rate thấp nhất | web_traffic.csv |
| Q5 | % order_items có promo | order_items.csv |
| Q6 | age_group có avg orders/customer cao nhất | customers + orders |
| Q7 | Region có tổng doanh thu cao nhất | orders + order_items + geography |
| Q8 | Payment method phổ biến nhất trong cancelled | orders.csv |
| Q9 | Size có tỷ lệ trả hàng cao nhất | returns + order_items + products |
| Q10 | Installment plan có avg payment cao nhất | payments.csv |

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../src/data/raw")

orders      = pd.read_csv(RAW / "orders.csv")
order_items = pd.read_csv(RAW / "order_items.csv")
products    = pd.read_csv(RAW / "products.csv")
customers   = pd.read_csv(RAW / "customers.csv")
geography   = pd.read_csv(RAW / "geography.csv")
payments    = pd.read_csv(RAW / "payments.csv")
returns     = pd.read_csv(RAW / "returns.csv")
web_traffic = pd.read_csv(RAW / "web_traffic.csv")
sales       = pd.read_csv(RAW / "sales.csv")

print("All data loaded")
for name, df in [("orders", orders), ("order_items", order_items), ("products", products),
                 ("customers", customers), ("geography", geography), ("payments", payments),
                 ("returns", returns), ("web_traffic", web_traffic), ("sales", sales)]:
    print(f"   {name:15s}: {len(df):>8,} rows")

E:\temp\ipykernel_14100\1889742396.py:8: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(RAW / "order_items.csv")


All data loaded
   orders         :  646,945 rows
   order_items    :  714,669 rows
   products       :    2,412 rows
   customers      :  121,930 rows
   geography      :   39,948 rows
   payments       :  646,945 rows
   returns        :   39,939 rows
   web_traffic    :    3,652 rows
   sales          :    3,833 rows


## Q1 — Median inter-order gap
**Câu hỏi:** Trong số các khách hàng có **nhiều hơn một đơn hàng**, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu?

- A) 30 ngày  
- B) 90 ngày  
- C) 180 ngày  
- D) 365 ngày  

**Logic:**
1. Lọc chỉ lấy khách mua ≥ 2 lần
2. Với mỗi khách: sort theo ngày, tính diff giữa các lần mua liên tiếp
3. Lấy median của tất cả gap đó

In [3]:
# Q1 — parse date chỉ khi cần, dùng format cụ thể
order_dates = pd.to_datetime(orders["order_date"], format="%Y-%m-%d", errors="coerce")
orders_tmp = orders.copy()
orders_tmp["order_date"] = order_dates

order_counts = orders_tmp.groupby("customer_id")["order_id"].count()
multi_buyers = order_counts[order_counts > 1].index

df_multi = orders_tmp[orders_tmp["customer_id"].isin(multi_buyers)].copy()
df_multi = df_multi.sort_values(["customer_id", "order_date"])
df_multi["prev_date"] = df_multi.groupby("customer_id")["order_date"].shift(1)
df_multi["gap_days"]  = (df_multi["order_date"] - df_multi["prev_date"]).dt.days

gaps = df_multi.dropna(subset=["gap_days"])["gap_days"]

median_gap = gaps.median()
mean_gap   = gaps.mean()
print(f"Median gap: {median_gap:.1f} ngay")
print(f"Mean gap:   {mean_gap:.1f} ngay")
print(f"Percentiles: 25th={gaps.quantile(0.25):.0f}d  50th={gaps.quantile(0.5):.0f}d  75th={gaps.quantile(0.75):.0f}d")

opts = {30: "A", 90: "B", 180: "C", 365: "D"}
closest = min(opts.keys(), key=lambda x: abs(x - median_gap))
print(f"DAP AN Q1: {opts[closest]}) {closest} ngay  (actual median = {median_gap:.1f}d)")

Median gap: 144.0 ngay
Mean gap:   285.6 ngay
Percentiles: 25th=46d  50th=144d  75th=357d
DAP AN Q1: C) 180 ngay  (actual median = 144.0d)


## Q2 — Segment có gross margin cao nhất
**Câu hỏi:** Phân khúc sản phẩm (`segment`) nào trong `products.csv` có **tỷ suất lợi nhuận gộp trung bình cao nhất**, với công thức `(price − cogs) / price`?

- A) Premium  
- B) Performance  
- C) Activewear  
- D) Standard  

**Logic:** Tính `(price - cogs) / price` cho từng sản phẩm → group by segment → mean

In [4]:
print("Unique segments:", products["segment"].unique().tolist())
print(f"Total products: {len(products):,}")
print()

# Tính gross margin từng sản phẩm
products["gross_margin"] = (products["price"] - products["cogs"]) / products["price"]

# Group by segment → mean
q2 = products.groupby("segment").agg(
    n_products   = ("product_id",    "count"),
    avg_price    = ("price",         "mean"),
    avg_cogs     = ("cogs",          "mean"),
    avg_margin   = ("gross_margin",  "mean")
).sort_values("avg_margin", ascending=False)

print("Gross margin by segment:")
print(q2.to_string(float_format="{:.4f}".format))

winner = q2["avg_margin"].idxmax()
print(f"\n>>> ĐÁP ÁN Q2: {winner}  (margin = {q2.loc[winner,'avg_margin']*100:.2f}%)")

Unique segments: ['Everyday', 'Performance', 'Balanced', 'Standard', 'All-weather', 'Premium', 'Trendy', 'Activewear']
Total products: 2,412

Gross margin by segment:
             n_products  avg_price  avg_cogs  avg_margin
segment                                                 
Standard            262  2928.5923 2259.3710      0.3134
Premium             177  2387.6749 1928.2522      0.2854
All-weather         169  3864.7461 3008.5220      0.2842
Activewear          598  2598.0960 2034.7554      0.2656
Performance         347  6572.8455 5095.1003      0.2636
Balanced            306  9230.2412 7230.8089      0.2580
Trendy              148  2212.7851 1745.3272      0.2408
Everyday            405  7549.1877 6007.4981      0.2363

>>> ĐÁP ÁN Q2: Standard  (margin = 31.34%)


## Q3 — Lý do trả hàng phổ biến nhất cho Streetwear
**Câu hỏi:** Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục **Streetwear** (join `returns` với `products` theo `product_id`), lý do trả hàng nào xuất hiện nhiều nhất?

- A) defective  
- B) wrong_size  
- C) changed_mind  
- D) not_as_described  

**Logic:** Filter products category=Streetwear → merge với returns → value_counts return_reason

In [5]:
print("Unique categories:", products["category"].unique().tolist())
print("Unique return_reason:", returns["return_reason"].unique().tolist())
print()

# Lấy product_id của Streetwear
sw_ids = products[products["category"].str.strip().str.lower() == "streetwear"]["product_id"]
print(f"Streetwear products: {len(sw_ids):,}")

# Join returns với Streetwear products
ret_sw = returns[returns["product_id"].isin(sw_ids)]
print(f"Returns for Streetwear: {len(ret_sw):,}  (out of {len(returns):,} total returns)")
print()

# Đếm lý do
q3 = ret_sw["return_reason"].value_counts()
print("Return reasons for Streetwear:")
for reason, cnt in q3.items():
    bar = "█" * int(cnt / q3.max() * 30)
    print(f"  {reason:20s}: {cnt:5,}  ({cnt/len(ret_sw)*100:5.1f}%)  {bar}")

winner = q3.idxmax()
print(f"\n>>> ĐÁP ÁN Q3: {winner}")

Unique categories: ['Streetwear', 'Casual', 'Outdoor', 'GenZ']
Unique return_reason: ['late_delivery', 'wrong_size', 'defective', 'changed_mind', 'not_as_described']

Streetwear products: 1,320
Returns for Streetwear: 21,799  (out of 39,939 total returns)

Return reasons for Streetwear:
  wrong_size          : 7,626  ( 35.0%)  ██████████████████████████████
  defective           : 4,330  ( 19.9%)  █████████████████
  not_as_described    : 3,854  ( 17.7%)  ███████████████
  changed_mind        : 3,830  ( 17.6%)  ███████████████
  late_delivery       : 2,159  (  9.9%)  ████████

>>> ĐÁP ÁN Q3: wrong_size


## Q4 — Traffic source có bounce_rate THẤP NHẤT
**Câu hỏi:** Trong `web_traffic.csv`, nguồn truy cập (`traffic_source`) nào có **tỷ lệ thoát trung bình (bounce_rate) thấp nhất** trên tất cả các ngày?

- A) organic_search  
- B) paid_search  
- C) email_campaign  
- D) social_media  

**Logic:** Group web_traffic by traffic_source → mean(bounce_rate) → idxmin()

In [8]:
print("Unique traffic_source:", web_traffic["traffic_source"].unique().tolist())

# Parse date if not already datetime
web_traffic["date"] = pd.to_datetime(web_traffic["date"], errors="coerce")
print(f"Date range: {web_traffic['date'].min().date()} → {web_traffic['date'].max().date()}")
print()

q4 = web_traffic.groupby("traffic_source").agg(
    n_days          = ("date",         "count"),
    avg_bounce_rate = ("bounce_rate",  "mean"),
    avg_sessions    = ("sessions",     "mean")
).sort_values("avg_bounce_rate")

print("Bounce rate by traffic source (thấp nhất trước):")
for src, row in q4.iterrows():
    arrow = " ◀ THẤP NHẤT" if src == q4["avg_bounce_rate"].idxmin() else ""
    print(f"  {src:20s}: {row['avg_bounce_rate']*100:6.2f}%  ({row['n_days']:,} ngày){arrow}")

winner = q4["avg_bounce_rate"].idxmin()
print(f"ĐÁP ÁN Q4: {winner}  (bounce_rate = {q4.loc[winner,'avg_bounce_rate']*100:.2f}%)")

Unique traffic_source: ['organic_search', 'direct', 'referral', 'social_media', 'paid_search', 'email_campaign']
Date range: 2013-01-01 → 2022-12-31

Bounce rate by traffic source (thấp nhất trước):
  email_campaign      :   0.45%  (505.0 ngày) ◀ THẤP NHẤT
  social_media        :   0.45%  (632.0 ngày)
  paid_search         :   0.45%  (784.0 ngày)
  referral            :   0.45%  (375.0 ngày)
  organic_search      :   0.45%  (1,090.0 ngày)
  direct              :   0.45%  (266.0 ngày)
ĐÁP ÁN Q4: email_campaign  (bounce_rate = 0.45%)


## Q5 — % order_items có áp dụng khuyến mãi
**Câu hỏi:** Tỷ lệ phần trăm các dòng trong `order_items.csv` có áp dụng khuyến mãi (tức là `promo_id` **không null**) xấp xỉ là bao nhiêu?

- A) 12%  
- B) 25%  
- C) 39%  
- D) 54%  

**Logic:** Đếm rows có `promo_id IS NOT NULL` / tổng rows × 100

In [9]:
print("Columns:", order_items.columns.tolist())
print(f"Total rows: {len(order_items):,}")
print()

total   = len(order_items)
has_p1  = order_items["promo_id"].notna().sum()
pct_p1  = has_p1 / total * 100

print(f"promo_id NOT NULL:  {has_p1:,}  →  {pct_p1:.2f}%")

# Kiểm tra promo_id_2 nếu có
if "promo_id_2" in order_items.columns:
    has_p2    = order_items["promo_id_2"].notna().sum()
    has_any   = (order_items["promo_id"].notna() | order_items["promo_id_2"].notna()).sum()
    pct_any   = has_any / total * 100
    print(f"promo_id_2 NOT NULL:{has_p2:,}  →  {has_p2/total*100:.2f}%")
    print(f"ANY promo (p1 OR p2):{has_any:,}  →  {pct_any:.2f}%")

print()
# Map đến đáp án
# Câu hỏi chỉ hỏi promo_id (không phải promo_id_2)
opts_map = [(12, "A"), (25, "B"), (39, "C"), (54, "D")]
closest_opt = min(opts_map, key=lambda x: abs(x[0] - pct_p1))
print(f">>> ĐÁP ÁN Q5: {closest_opt[1]}) {closest_opt[0]}%  (actual = {pct_p1:.1f}%)")

Columns: ['order_id', 'product_id', 'quantity', 'unit_price', 'discount_amount', 'promo_id', 'promo_id_2']
Total rows: 714,669

promo_id NOT NULL:  276,316  →  38.66%
promo_id_2 NOT NULL:206  →  0.03%
ANY promo (p1 OR p2):276,316  →  38.66%

>>> ĐÁP ÁN Q5: C) 39%  (actual = 38.7%)


## Q6 — age_group có avg orders/customer cao nhất
**Câu hỏi:** Trong `customers.csv`, xét các khách hàng có `age_group` khác null, nhóm tuổi nào có **số đơn hàng trung bình trên mỗi khách hàng** cao nhất? (tổng số đơn / số khách hàng trong nhóm)

- A) 55+  
- B) 25–34  
- C) 35–44  
- D) 45–54  

**Logic:** Join customers (age_group != null) với orders → group by age_group → count(orders) / nunique(customers)

In [10]:
print("Unique age_group:", customers["age_group"].dropna().unique().tolist())
print(f"Customers with age_group: {customers['age_group'].notna().sum():,} / {len(customers):,}")
print()

# Chỉ lấy customers có age_group
cust_age = customers[customers["age_group"].notna()][["customer_id", "age_group"]]

# Join với orders (tất cả orders, không filter status)
merged = orders.merge(cust_age, on="customer_id", how="inner")

q6 = merged.groupby("age_group").agg(
    total_orders    = ("order_id",     "count"),
    unique_customers= ("customer_id",  "nunique")
).reset_index()
q6["avg_orders_per_cust"] = q6["total_orders"] / q6["unique_customers"]
q6 = q6.sort_values("avg_orders_per_cust", ascending=False)

print("avg orders per customer by age_group:")
for _, row in q6.iterrows():
    arrow = " ◀ CAO NHẤT" if row["age_group"] == q6.iloc[0]["age_group"] else ""
    print(f"  {row['age_group']:8s}: {row['avg_orders_per_cust']:.3f}  "
          f"({row['total_orders']:,} orders / {row['unique_customers']:,} customers){arrow}")

winner = q6.iloc[0]["age_group"]
print(f"\n>>> ĐÁP ÁN Q6: {winner}")

Unique age_group: ['35-44', '45-54', '18-24', '55+', '25-34']
Customers with age_group: 121,930 / 121,930

avg orders per customer by age_group:
  55+     : 7.269  (72,760 orders / 10,010 customers) ◀ CAO NHẤT
  45-54   : 7.220  (124,138 orders / 17,193 customers)
  35-44   : 7.206  (170,368 orders / 23,642 customers)
  25-34   : 7.112  (190,622 orders / 26,802 customers)
  18-24   : 7.069  (89,057 orders / 12,599 customers)

>>> ĐÁP ÁN Q6: 55+


## Q7 — Region có tổng doanh thu cao nhất
**Câu hỏi:** Vùng (`region`) nào trong `geography.csv` tạo ra **tổng doanh thu cao nhất** trong `sales_train.csv`?

- A) West  
- B) Central  
- C) East  
- D) Cả ba vùng có doanh thu xấp xỉ bằng nhau  

**Logic quan trọng:** `sales.csv` chỉ có `Date, Revenue, COGS` — không có `region`. Phải tính revenue từ `order_items` (quantity × unit_price), join qua `orders.zip → geography.region`, trong khoảng thời gian của sales_train (2012-07-04 → 2022-12-31).

In [12]:
print("Unique regions:", geography["region"].unique().tolist())

# Parse dates
sales["Date"] = pd.to_datetime(sales["Date"], errors="coerce")
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
print(f"Sales train period: {sales['Date'].min().date()} → {sales['Date'].max().date()}")
print()

# Bước 1: Line revenue từ order_items
order_items["line_revenue"] = order_items["quantity"] * order_items["unit_price"]

# Bước 2: Map orders → region qua zip
geo_zip = geography[["zip","region"]].drop_duplicates("zip")
orders_geo = orders.merge(geo_zip, on="zip", how="left")
print(f"Orders matched to region: {orders_geo['region'].notna().sum():,} / {len(orders_geo):,}")

# Bước 3: Filter trong khoảng sales_train
train_start = sales["Date"].min()
train_end   = sales["Date"].max()
orders_train = orders_geo[
    (orders_geo["order_date"] >= train_start) &
    (orders_geo["order_date"] <= train_end)
][["order_id","region"]]

# Bước 4: Join order_items → region
oi_region = order_items.merge(orders_train, on="order_id", how="inner")

q7 = oi_region.groupby("region")["line_revenue"].sum().sort_values(ascending=False)
total_rev = q7.sum()

print("Total revenue by region (training period):")
for reg, val in q7.items():
    pct   = val / total_rev * 100
    bar   = "█" * int(pct / 2)
    arrow = " ◄ CAO NHẤT" if reg == q7.idxmax() else ""
    print(f"  {str(reg):12s}: {val/1e9:8.2f} tỷ VND  ({pct:5.1f}%)  {bar}{arrow}")

winner   = q7.idxmax()
max_pct  = q7.max() / total_rev * 100
if q7.max() / q7.min() < 1.1:
    print("3 vùng xấp xỉ nhau → D)")
print(f"ĐÁP ÁN Q7: {winner}  ({max_pct:.1f}% share)")

Unique regions: ['East', 'Central', 'West']
Sales train period: 2012-07-04 → 2022-12-31

Orders matched to region: 646,945 / 646,945
Total revenue by region (training period):
  East        :     7.64 tỷ VND  ( 46.5%)  ███████████████████████ ◄ CAO NHẤT
  Central     :     4.94 tỷ VND  ( 30.1%)  ███████████████
  West        :     3.85 tỷ VND  ( 23.4%)  ███████████
ĐÁP ÁN Q7: East  (46.5% share)


## Q8 — Payment method phổ biến nhất trong đơn CANCELLED
**Câu hỏi:** Trong các đơn hàng có `order_status = 'cancelled'` trong `orders.csv`, **phương thức thanh toán** nào được sử dụng nhiều nhất?

- A) credit_card  
- B) cod  
- C) paypal  
- D) bank_transfer  

**Logic:** Filter orders where order_status='cancelled' → value_counts(payment_method)

In [13]:
print("Unique order_status:", orders["order_status"].unique().tolist())
print("Unique payment_method:", orders["payment_method"].unique().tolist())
print()

cancelled = orders[orders["order_status"] == "cancelled"]
print(f"Cancelled orders: {len(cancelled):,} / {len(orders):,} ({len(cancelled)/len(orders)*100:.1f}%)")
print()

q8 = cancelled["payment_method"].value_counts()
print("Payment method in CANCELLED orders:")
for pm, cnt in q8.items():
    bar   = "█" * int(cnt / q8.max() * 30)
    arrow = " ◀ NHIỀU NHẤT" if pm == q8.idxmax() else ""
    print(f"  {pm:15s}: {cnt:6,}  ({cnt/len(cancelled)*100:5.1f}%)  {bar}{arrow}")

# So sánh với toàn bộ orders để thấy sự chênh lệch
print()
print("So sánh với toàn bộ orders:")
all_pm = orders["payment_method"].value_counts(normalize=True) * 100
for pm in q8.index:
    delta = q8[pm]/len(cancelled)*100 - all_pm.get(pm, 0)
    sign  = "↑" if delta > 0 else "↓"
    print(f"  {pm:15s}: cancelled={q8[pm]/len(cancelled)*100:.1f}%  all={all_pm.get(pm,0):.1f}%  {sign}{abs(delta):.1f}pp")

winner = q8.idxmax()
print(f"\n>>> ĐÁP ÁN Q8: {winner}")

Unique order_status: ['delivered', 'returned', 'shipped', 'cancelled', 'paid', 'created']
Unique payment_method: ['credit_card', 'cod', 'paypal', 'apple_pay', 'bank_transfer']

Cancelled orders: 59,462 / 646,945 (9.2%)

Payment method in CANCELLED orders:
  credit_card    : 28,452  ( 47.8%)  ██████████████████████████████ ◀ NHIỀU NHẤT
  cod            : 15,468  ( 26.0%)  ████████████████
  paypal         :  7,817  ( 13.1%)  ████████
  apple_pay      :  5,190  (  8.7%)  █████
  bank_transfer  :  2,535  (  4.3%)  ██

So sánh với toàn bộ orders:
  credit_card    : cancelled=47.8%  all=55.1%  ↓7.2pp
  cod            : cancelled=26.0%  all=14.9%  ↑11.1pp
  paypal         : cancelled=13.1%  all=15.0%  ↓1.9pp
  apple_pay      : cancelled=8.7%  all=10.0%  ↓1.3pp
  bank_transfer  : cancelled=4.3%  all=5.0%  ↓0.7pp

>>> ĐÁP ÁN Q8: credit_card


## Q9 — Size có tỷ lệ trả hàng cao nhất
**Câu hỏi:** Trong bốn kích thước sản phẩm **(S, M, L, XL)**, kích thước nào có **tỷ lệ trả hàng cao nhất**, được định nghĩa là số bản ghi trong `returns` chia cho số dòng trong `order_items` (join với `products` theo `product_id`)?

- A) S  
- B) M  
- C) L  
- D) XL  

**Logic:**
- **Tử số:** COUNT(returns records) với product.size = X
- **Mẫu số:** COUNT(order_items rows) với product.size = X  
- Lưu ý: returns dùng `return_quantity` nhưng đề bài hỏi "số bản ghi" → dùng COUNT rows, không phải SUM quantity

In [14]:
sizes_of_interest = ["S", "M", "L", "XL"]
print("Unique sizes:", products["size"].dropna().unique().tolist())
print()

# Mẫu số: order_items rows per size
oi_sz = order_items.merge(products[["product_id","size"]], on="product_id", how="left")
oi_by_sz = (oi_sz[oi_sz["size"].isin(sizes_of_interest)]
             .groupby("size").size()
             .rename("n_order_items"))

# Tử số: returns records per size
ret_sz = returns.merge(products[["product_id","size"]], on="product_id", how="left")
ret_by_sz = (ret_sz[ret_sz["size"].isin(sizes_of_interest)]
              .groupby("size").size()
              .rename("n_returns"))

q9 = pd.DataFrame({"n_returns": ret_by_sz, "n_order_items": oi_by_sz}).fillna(0)
q9["return_rate"] = q9["n_returns"] / q9["n_order_items"]
q9 = q9.sort_values("return_rate", ascending=False)

print("Return rate by size (cao nhất trước):")
for size, row in q9.iterrows():
    bar   = "█" * int(row["return_rate"] / q9["return_rate"].max() * 25)
    arrow = " ◀ CAO NHẤT" if size == q9["return_rate"].idxmax() else ""
    print(f"  {size}: {row['return_rate']*100:6.3f}%  "
          f"returns={int(row['n_returns']):,}  items={int(row['n_order_items']):,}  {bar}{arrow}")

winner = q9["return_rate"].idxmax()
opts   = {"S":"A","M":"B","L":"C","XL":"D"}
print(f"\n>>> ĐÁP ÁN Q9: {opts.get(winner,'?')}) {winner}  "
      f"(return_rate = {q9.loc[winner,'return_rate']*100:.3f}%)")

Unique sizes: ['S', 'M', 'L', 'XL']

Return rate by size (cao nhất trước):
  S:  5.652%  returns=9,723  items=172,042  █████████████████████████ ◀ CAO NHẤT
  L:  5.625%  returns=9,741  items=173,174  ████████████████████████
  M:  5.566%  returns=9,820  items=176,428  ████████████████████████
  XL:  5.520%  returns=10,655  items=193,025  ████████████████████████

>>> ĐÁP ÁN Q9: A) S  (return_rate = 5.652%)


## Q10 — Installment plan có avg payment_value cao nhất
**Câu hỏi:** Trong `payments.csv`, **kế hoạch trả góp** nào có **giá trị thanh toán trung bình trên mỗi đơn hàng** cao nhất?

- A) 1 kỳ (trả một lần)  
- B) 3 kỳ  
- C) 6 kỳ  
- D) 12 kỳ  

**Logic:** Group payments by installments → mean(payment_value) → chỉ xét 4 mức 1/3/6/12

In [15]:
print("Unique installments:", sorted(payments["installments"].dropna().unique().tolist()))
print(f"Total payment rows: {len(payments):,}")
print()

# Tất cả mức (để hiểu data)
q10_all = payments.groupby("installments")["payment_value"].agg(
    count="count", mean="mean", median="median"
).sort_values("mean", ascending=False)
print("Avg payment_value by ALL installment levels (cao nhất trước):")
print(q10_all.to_string(float_format="{:,.0f}".format))
print()

# Chỉ các mức trong đáp án: 1, 3, 6, 12
opts_inst = {1:"A", 3:"B", 6:"C", 12:"D"}
q10_opts  = payments[payments["installments"].isin(opts_inst.keys())]
q10_grp   = q10_opts.groupby("installments")["payment_value"].mean().sort_values(ascending=False)

print("Avg payment_value — chỉ 4 mức đáp án:")
for inst, val in q10_grp.items():
    arrow = " ◀ CAO NHẤT" if inst == q10_grp.idxmax() else ""
    print(f"  {int(inst):2d} kỳ ({opts_inst[int(inst)]}): {val:>12,.2f} VND{arrow}")

winner     = int(q10_grp.idxmax())
ans_letter = opts_inst[winner]
print(f"\n>>> ĐÁP ÁN Q10: {ans_letter}) {winner} kỳ  "
      f"(avg = {q10_grp[winner]:,.2f} VND)")

Unique installments: [1, 2, 3, 6, 12]
Total payment rows: 646,945

Avg payment_value by ALL installment levels (cao nhất trước):
               count   mean  median
installments                       
6             109910 24,447  17,452
3             218949 24,400  17,367
12             54126 24,246  17,337
1             262866 24,113  17,089
2               1094    708     722

Avg payment_value — chỉ 4 mức đáp án:
   6 kỳ (C):    24,446.65 VND ◀ CAO NHẤT
   3 kỳ (B):    24,399.64 VND
  12 kỳ (D):    24,245.77 VND
   1 kỳ (A):    24,113.27 VND

>>> ĐÁP ÁN Q10: C) 6 kỳ  (avg = 24,446.65 VND)


## TỔNG KẾT — 10 ĐÁP ÁN MCQ
Chạy cell dưới để in bảng tổng hợp tất cả đáp án.

In [16]:
# ─── Re-compute all answers compactly ───────────────────────────────────────

# Q1
gaps_q1 = orders.sort_values(["customer_id","order_date"])
gaps_q1["prev"] = gaps_q1.groupby("customer_id")["order_date"].shift(1)
gaps_q1["gap"]  = (gaps_q1["order_date"] - gaps_q1["prev"]).dt.days
multi = orders.groupby("customer_id")["order_id"].count()
med   = gaps_q1[gaps_q1["customer_id"].isin(multi[multi>1].index)].dropna(subset=["gap"])["gap"].median()
A1 = f"B) 90 ngày" if 45<med<=135 else (f"A) 30 ngày" if med<=45 else (f"C) 180 ngày" if med<=272 else "D) 365 ngày"))

# Q2
products["gm"] = (products["price"] - products["cogs"]) / products["price"]
A2 = products.groupby("segment")["gm"].mean().idxmax()

# Q3
sw = products[products["category"].str.lower()=="streetwear"]["product_id"]
A3 = returns[returns["product_id"].isin(sw)]["return_reason"].value_counts().idxmax()

# Q4
A4 = web_traffic.groupby("traffic_source")["bounce_rate"].mean().idxmin()

# Q5
pct5 = order_items["promo_id"].notna().mean()*100
A5 = min([(12,"A"),(25,"B"),(39,"C"),(54,"D")], key=lambda x:abs(x[0]-pct5))
A5 = f"{A5[1]}) {A5[0]}%  (actual={pct5:.1f}%)"

# Q6
ca = customers[customers["age_group"].notna()][["customer_id","age_group"]]
mg = orders.merge(ca,on="customer_id",how="inner").groupby("age_group")
A6 = (mg["order_id"].count() / mg["customer_id"].nunique()).idxmax()

# Q7
order_items["lr"] = order_items["quantity"]*order_items["unit_price"]
og = orders.merge(geography[["zip","region"]].drop_duplicates("zip"),on="zip",how="left")
ot = og[(og["order_date"]>=sales["Date"].min())&(og["order_date"]<=sales["Date"].max())][["order_id","region"]]
rv = order_items.merge(ot,on="order_id",how="inner").groupby("region")["lr"].sum()
A7 = rv.idxmax(); mp = rv.max()/rv.sum()*100
A7 = f"{A7} ({mp:.1f}%)" if mp>37 else f"D) Cả ba xấp xỉ ({mp:.1f}%)"

# Q8
A8 = orders[orders["order_status"]=="cancelled"]["payment_method"].value_counts().idxmax()

# Q9
ois = order_items.merge(products[["product_id","size"]],on="product_id",how="left")
rts = returns.merge(products[["product_id","size"]],on="product_id",how="left")
sz  = ["S","M","L","XL"]
nb  = ois[ois["size"].isin(sz)].groupby("size").size()
nr  = rts[rts["size"].isin(sz)].groupby("size").size()
rr  = (nr/nb).dropna(); A9w = rr.idxmax()
A9  = {"S":"A","M":"B","L":"C","XL":"D"}.get(A9w,"?")+f") {A9w}  ({rr[A9w]*100:.3f}%)"

# Q10
oi10 = {1:"A",3:"B",6:"C",12:"D"}
pg   = payments[payments["installments"].isin(oi10.keys())].groupby("installments")["payment_value"].mean()
A10w = int(pg.idxmax()); A10 = f"{oi10[A10w]}) {A10w} kỳ  ({pg[A10w]:,.0f} VND)"

# ─── Print summary ────────────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════════════════╗")
print("║           TỔNG KẾT ĐÁP ÁN MCQ — DATATHON 2026 ROUND 1          ║")
print("╠══════════════════════════════════════════════════════════════════╣")
rows = [
    ("Q1",  "Median inter-order gap",                    A1),
    ("Q2",  "Segment gross margin cao nhất",             A2),
    ("Q3",  "Lý do trả hàng Streetwear nhiều nhất",     A3),
    ("Q4",  "Traffic source bounce_rate thấp nhất",     A4),
    ("Q5",  "% order_items có promo",                   A5),
    ("Q6",  "age_group avg orders/cust cao nhất",       A6),
    ("Q7",  "Region revenue cao nhất",                   A7),
    ("Q8",  "Payment method trong cancelled orders",    A8),
    ("Q9",  "Size tỷ lệ trả hàng cao nhất",            A9),
    ("Q10", "Installment avg payment cao nhất",         A10),
]
for q, desc, ans in rows:
    print(f"║ {q:<3s} │ {desc:<38s} │ {ans:<20s} ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print(f"\nTổng điểm tối đa Part 1: 20 pts (2 pts × 10 câu, không trừ điểm sai)")

╔══════════════════════════════════════════════════════════════════╗
║           TỔNG KẾT ĐÁP ÁN MCQ — DATATHON 2026 ROUND 1          ║
╠══════════════════════════════════════════════════════════════════╣
║ Q1  │ Median inter-order gap                 │ C) 180 ngày          ║
║ Q2  │ Segment gross margin cao nhất          │ Standard             ║
║ Q3  │ Lý do trả hàng Streetwear nhiều nhất   │ wrong_size           ║
║ Q4  │ Traffic source bounce_rate thấp nhất   │ email_campaign       ║
║ Q5  │ % order_items có promo                 │ C) 39%  (actual=38.7%) ║
║ Q6  │ age_group avg orders/cust cao nhất     │ 55+                  ║
║ Q7  │ Region revenue cao nhất                │ East (46.5%)         ║
║ Q8  │ Payment method trong cancelled orders  │ credit_card          ║
║ Q9  │ Size tỷ lệ trả hàng cao nhất           │ A) S  (5.652%)       ║
║ Q10 │ Installment avg payment cao nhất       │ C) 6 kỳ  (24,447 VND) ║
╚══════════════════════════════════════════════════════════════════╝

Tổ